# Session Viewer: force density with NumPy
Solve a small cable net, then display the result with `session_viewer`. Fixed nodes are blue, free nodes are pink, and edge width increases with axial force.

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps
%pip install -q https://brg-teaching.github.io/Teaching/CAS/website/session-viewer/session_py-0.260.0-py3-none-any.whl

In [ ]:
import numpy as np
from IPython.display import display
from compas.geometry import Line as CompasLine
from compas_notebook import Viewer
from compas_notebook.config import Config
from session_py import Color, Line, Point, Session, show
from session_py.viewer import supports_webgpu

## Net: five free nodes, four fixed nodes

In [ ]:
edges = [(0, 5), (1, 6), (2, 7), (3, 8), (0, 3), (0, 1), (1, 2), (2, 3), (3, 4), (0, 4), (1, 4), (2, 4)]
xyz_fixed = np.array([[-2, 0, 0], [0, -2, 1], [2, 0, 0], [0, 2, 1]], dtype=float)
nfree = 5
q = np.array([2.0] * 4 + [1.0] * 8)
p = np.zeros((nfree, 3))
p[:, 2] = -0.5

## Connectivity and force-density matrices

In [ ]:
C = np.zeros((len(edges), nfree + len(xyz_fixed)))
for j, (i, k) in enumerate(edges):
    C[j, i], C[j, k] = 1, -1
Cn, Cf = C[:, :nfree], C[:, nfree:]
Q = np.diag(q)

## Solve for the free nodes

In [ ]:
xyz_free = np.linalg.solve(Cn.T @ Q @ Cn, p - Cn.T @ Q @ Cf @ xyz_fixed)
xyz = np.vstack([xyz_free, xyz_fixed])
forces = q * np.linalg.norm(C @ xyz, axis=1)
xyz_free.round(3), forces.round(3)

## Build a session
A `Session` contains named points and lines. Their colours and widths control their appearance in the viewer.

In [ ]:
session = Session("force_density")
blue = Color(48 / 255, 87 / 255, 177 / 255)
pink = Color(220 / 255, 75 / 255, 141 / 255)
nodes = []
for index, coordinates in enumerate(xyz):
    fixed = index >= nfree
    node = Point(*coordinates, name=f"n{index}")
    node.width = 7.0 if fixed else 5.0
    node.pointcolor = blue if fixed else pink
    session.add_point(node, None)
    nodes.append(node)
for index, ((start, end), force) in enumerate(zip(edges, forces)):
    line = Line.from_points(nodes[start], nodes[end])
    line.name = f"e{index}"
    line.width = 1.0 + 3.0 * float(force) / forces.max()
    line.linecolor = pink
    session.add_line(line, None)

## Display
`show(session)` embeds the interactive WebGPU viewer inside this cell's output. Browsers without WebGPU use the existing COMPAS viewer instead.

In [ ]:
if await supports_webgpu():
    display(show(session, height=420))
else:
    config = Config()
    config.view.width, config.view.height = 700, 360
    config.view.camera.position = [0, -6, 4]
    config.view.camera.target = [0, 0, 0]
    viewer = Viewer(config=config)
    for start, end in edges:
        viewer.scene.add(CompasLine(xyz[start], xyz[end]), color="#dc4b8d")
    viewer.show()